# Smart Dirt Buildup Detection for Conveyor Belt Systems

**Authors:** Erick Chauke, Dr Milka Madahana, & Dr John Ekoru  
**Dataset:** [IEEE DataPort](https://ieee-dataport.org/open-access/dirt-buildup-belt-conveyor-structures)

---

## Introduction

Mining operations rely on long conveyor belt systems to transport ore across distances. Over time, dirt and material from the ore accumulates on the belt structures, specifically on rollers and frames. When this buildup goes undetected, rollers seize up, belts drift sideways, and in serious cases fires start. A single unplanned stoppage costs a mine tens of thousands of dollars per hour.

This notebook addresses that problem by training a model to classify images of conveyor belt structures as either clean or carrying dirt buildup. The dataset contains 388 labelled photographs sourced from a real mining environment.

The approach taken in this notebook:

- Exploring and understanding the dataset before building anything
- Splitting data correctly to prevent leakage, then applying preprocessing and augmentation
- Establishing a scratch-built baseline model
- Training and fine-tuning two pretrained models, ResNet-50 and EfficientNet-B0
- Evaluating all models against the benchmark set by Santos et al. (2020), who achieved 89.75% accuracy and an F1-score of 0.8773 on a similar task
- Interpreting model decisions using Grad-CAM heatmaps and analysing failure cases


### Loading Libraries

In [2]:
# Install required libraries quietly
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["timm", "albumentations", "scikit-learn", "matplotlib", "seaborn"]:
    install(pkg)

# Core libraries
import random                                    # random number generation
import numpy as np                               # numerical operations
import torch                                     # neural network framework
import timm                                      # pretrained model library
import albumentations as A                       # image augmentation
import matplotlib.pyplot as plt                  # plotting and visualisation
import seaborn as sns                            # statistical charts
from sklearn.model_selection import train_test_split  # dataset splitting


SEED = 42  # any fixed number works, 42 is a common convention in data science

# Fix all random sources so every run produces the same results
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# Use GPU if available, otherwise fall back to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cpu
